Data source: https://www.kaggle.com/competitions/iaa-ai-assisted-hackathon-2026-part-1

In [0]:
# Seaborn and openpyxl (needed for Excel exports) are not installed in this environment
%pip install seaborn openpyxl

In [0]:
# Import libraries:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

#### Loading Data

In [0]:
import os

# Load every .parquet file in the data folder of the current branch
data_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/data"
parquet_files = sorted(f for f in os.listdir(data_dir) if f.lower().endswith(".parquet"))

dataframes = {}   # dataset name -> Spark DataFrame
row_counts = {}   # dataset name -> total row count

for file_name in parquet_files:
    name = file_name.replace(".parquet", "")
    df = spark.read.parquet(os.path.join(data_dir, file_name))
    dataframes[name] = df
    globals()[f"df_{name}"] = df   # each dataset is also its own variable, e.g. df_claims

# Report how many files were imported and their names
print(f"Imported {len(dataframes)} parquet files from {data_dir}:\n")
for name, df in dataframes.items():
    row_counts[name] = df.count()
    print(f"  df_{name:<18} | {row_counts[name]:>10,} rows | {len(df.columns)} columns")

In [0]:
import pandas as pd

# Folder for all Excel exports (created if it doesn't exist)
excel_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files"
os.makedirs(excel_dir, exist_ok=True)

def _sheet_name(name):
    r"""Excel sheet names: max 31 chars, cannot contain : \ / ? * [ ]"""
    return "".join(c for c in name if c not in ":\\/?*[]")[:31] or "sheet"

overview_path = os.path.join(excel_dir, "Data_overview_100_rows.xlsx")

with pd.ExcelWriter(overview_path, engine="openpyxl") as writer:
    for name, df in dataframes.items():
        sheet = _sheet_name(name)
        # Line at the top of the sheet stating the total number of rows
        summary = pd.DataFrame([[f"{name}: {row_counts[name]:,} total rows (showing first 100)"]])
        summary.to_excel(writer, sheet_name=sheet, index=False, header=False, startrow=0)
        # First 100 rows of the dataframe, a few rows below the summary line
        df.limit(100).toPandas().to_excel(writer, sheet_name=sheet, index=False, startrow=3)

print(f"Overview workbook saved to: {overview_path}")
print(f"Contains one sheet per dataset: {', '.join(dataframes.keys())}")

In [0]:
# Export only the claims dataframe in full to Excel.
# The other datasets are too large to be practical in Excel — use the
# Data_overview_100_rows.xlsx workbook to preview them instead.
claims_path = os.path.join(excel_dir, "claims.xlsx")
dataframes["claims"].toPandas().to_excel(claims_path, engine="openpyxl", index=False)
print(f"Saved claims.xlsx ({row_counts['claims']:,} rows) to: {excel_dir}")

In [0]:
from pyspark.sql import functions as F

# Build a summary workbook: an Overview sheet (one row per dataset) plus a sheet
# per dataset with column dtypes, missing values and describe statistics.
summary_path = os.path.join(excel_dir, "Data_Summary.xlsx")

overview_rows = []
per_dataset_stats = {}   # dataset name -> column-level stats DataFrame
shapes = {}              # dataset name -> (rows, columns)

for name, df in dataframes.items():
    n_rows = df.count()
    dtypes = dict(df.dtypes)
    cols = list(dtypes.keys())
    numeric_cols = [c for c, t in dtypes.items()
                    if t in ("int", "bigint", "smallint", "tinyint", "double", "float")
                    or t.startswith("decimal")]
    shapes[name] = (n_rows, len(cols))

    # Missing values per column (single aggregation pass)
    null_row = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).first()
    null_counts = {c: null_row[c] for c in cols}

    stats = pd.DataFrame({
        "column": cols,
        "dtype": [dtypes[c] for c in cols],
        "missing_values": [null_counts[c] for c in cols],
        "missing_%": [round(100.0 * null_counts[c] / n_rows, 2) if n_rows else 0.0 for c in cols],
    })

    # Describe statistics (count/mean/stddev/min/max) for numeric columns
    if numeric_cols:
        desc = (df.select(numeric_cols).describe()
                .toPandas().set_index("summary").T
                .apply(pd.to_numeric, errors="coerce"))
        stats = stats.merge(desc, left_on="column", right_index=True, how="left")
    per_dataset_stats[name] = stats

    total_missing = sum(null_counts.values())
    overview_rows.append({
        "dataframe": name,
        "rows": n_rows,
        "columns": len(cols),
        "numeric_columns": len(numeric_cols),
        "total_missing_values": total_missing,
        "missing_%_of_cells": round(100.0 * total_missing / (n_rows * len(cols)), 2) if n_rows and cols else 0.0,
    })

with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:
    overview = pd.DataFrame(overview_rows)
    overview.to_excel(writer, sheet_name="Overview", index=False)
    for name, stats in per_dataset_stats.items():
        sheet = _sheet_name(name)
        n_rows, n_cols = shapes[name]
        header = pd.DataFrame([[f"{name}: {n_rows:,} rows x {n_cols} columns"]])
        header.to_excel(writer, sheet_name=sheet, index=False, header=False, startrow=0)
        stats.to_excel(writer, sheet_name=sheet, index=False, startrow=2)

print(f"Summary workbook saved to: {summary_path}")
display(overview)

#### Create master policy files

In [0]:
# Loop through motor/contents/building to create 3 master policy files with 
# informtaion from the other files appended.
# Note that not all information will be carried through - e.g. if there are multiple
# claims for a policy row we can only bring through one line - if this happens we will 
# note if there are multiple perils and the total count of claims for that row
#
# NOTE: the addresses file has no customer_id column, so addresses is joined on
# address_id (carried on the policy files) instead.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Claim fields to take from the first claim (lowest claim_id) for each policy
CLAIM_FIRST_COLS = ["claim_id", "loss_date", "report_date", "status", "repairer_id", "third_party_id"]

# Weather join keys renamed so they don't clash with the policy file's own columns
wx = (dataframes["weather"]
      .withColumnRenamed("address_id", "_wx_address_id")
      .withColumnRenamed("year", "_wx_year"))

master_dataframes = {}   # e.g. {"policies_motor_master": DataFrame}

for line_name, pol_name in [("motor", "policies_motor"),
                            ("contents", "policies_contents"),
                            ("buildings", "policies_buildings")]:
    pol = dataframes[pol_name]

    # HUMAN DECISION (2026-10-20, may be revisited): keep only policies whose
    # renewal_decision is 'renewed' (~81-84% of rows per line). This drops
    # lapsed_by_customer / cancelled / declined_by_insurer / mid_term_change rows,
    # including short-period cancellations such as single-day policies.
    pol = pol.filter(F.col("renewal_decision") == "renewed")

    # ---- Claims: keep only this line, aggregate per policy_id and loss year ----
    # (aggregated before joining so the join can never fan out policy rows)
    cl = (dataframes["claims"]
          .filter(F.col("line") == line_name)
          .withColumn("_claim_year", F.year("loss_date"))
          .withColumn("_rn", F.row_number().over(
              Window.partitionBy("policy_id", "_claim_year").orderBy("claim_id"))))

    cl_agg = (cl.groupBy("policy_id", "_claim_year")
              .agg(
                  F.sum("paid_to_date").alias("paid_to_date"),
                  F.sum("incurred_to_date").alias("incurred_to_date"),
                  F.sum("recoveries").alias("recoveries"),
                  F.count(F.lit(1)).alias("n_claims"),
                  F.collect_set("peril").alias("_perils"),
                  *[F.max(F.when(F.col("_rn") == 1, F.col(c))).alias(c) for c in CLAIM_FIRST_COLS])
              .withColumn("peril",
                          F.when(F.size(F.col("_perils")) > 1, F.lit("multiple"))
                           .otherwise(F.element_at(F.col("_perils"), 1)))
              .drop("_perils")
              .withColumnRenamed("policy_id", "_cl_policy_id"))

    # ---- Master file: policies + weather + customers + addresses + claims ----
    master = (
        pol
        # weather at the policy's address for the policy year
        .join(wx,
              (F.col("address_id") == F.col("_wx_address_id")) &
              (F.year(F.col("period_start")) == F.col("_wx_year")),
              "left")
        .drop("_wx_address_id", "_wx_year")
        # customer details
        .join(dataframes["customers"], "customer_id", "left")
        # address details - joined on address_id (addresses has no customer_id)
        .join(dataframes["addresses"], "address_id", "left")
        # claims on this line whose loss year matches the policy year
        .join(cl_agg,
              (F.col("policy_id") == F.col("_cl_policy_id")) &
              (F.year(F.col("period_start")) == F.col("_claim_year")),
              "left")
        .drop("_cl_policy_id", "_claim_year")
        # claim_indicator: 1 if the policy row has any claims (n_claims >= 1), 0 otherwise
        .withColumn("claim_indicator", F.when(F.col("n_claims") >= 1, 1).otherwise(0))
        # incurred net of recoveries: aggregated incurred_to_date minus aggregated
        # recoveries (null for policy-years with no claims)
        .withColumn("incurred_net_of_recoveries", F.col("incurred_to_date") - F.col("recoveries"))
        # ultimate_claims: for now equal to incurred_to_date - recoveries; may be
        # revised later if claims are developed to ultimate (see AI_ASSISTANCE.md)
        .withColumn("ultimate_claims", F.col("incurred_to_date") - F.col("recoveries"))
        # earned_premium: the annualized written_premium earned over the days actually
        # in force, pro-rated by the number of days in the policy's calendar year.
        # DECISION / ASSUMPTION (2026-10-20, flagged in AI_ASSISTANCE.md): written_premium
        # is treated as the ANNUAL premium under the insurer's current pricing (evidence:
        # premium is unrelated to term length, corr ~0; the same policy pays ~99.6% as much
        # for a ~104-day period as for a following full year). Day counts are inclusive
        # of both endpoints, e.g. 2021-01-01 to 2021-12-31 = 365 days (366 in 2024).
        # earned_premium = written_premium x days_in_force / days_in_calendar_year
        .withColumn("_days_total", F.datediff("period_end", "period_start") + 1)
        .withColumn("_year_end", F.make_date(F.year("period_start"), F.lit(12), F.lit(31)))
        .withColumn("_days_earned", F.least(F.col("_days_total"),              # days in force during
                                            F.datediff("_year_end", "period_start") + 1))  # this calendar year
        .withColumn("_year_days", F.datediff("_year_end",
                                             F.make_date(F.year("period_start"), F.lit(1), F.lit(1))) + 1)
        .withColumn("earned_premium",
                    F.round(F.col("written_premium") * F.col("_days_earned") / F.col("_year_days"), 2))
        .drop("_days_total", "_year_end", "_days_earned", "_year_days")
    )

    master_name = f"{pol_name}_master"
    master_dataframes[master_name] = master
    globals()[f"df_{master_name}"] = master   # e.g. df_policies_motor_master

    # Sanity checks in one pass: master row count must equal the renewed policy rows (no fan-out)
    n_pol_all = dataframes[pol_name].count()
    n_pol = pol.count()
    s = master.agg(
        F.count(F.lit(1)).alias("rows"),
        F.sum(F.when(F.col("n_claims").isNotNull(), 1).otherwise(0)).alias("with_claims"),
        F.sum("claim_indicator").alias("claim_ind_total"),
        F.sum(F.when(F.col("earned_premium").isNull(), 1).otherwise(0)).alias("earned_null"),
        F.round(F.sum("incurred_to_date") - F.sum("recoveries"), 0).alias("chk_net"),
        F.round(F.sum("ultimate_claims"), 0).alias("ult_total"),
        F.sum(F.when(F.col("earned_premium") > F.col("written_premium"), 1).otherwise(0)).alias("earned_over"),
        F.sum(F.when(F.col("hail_days").isNotNull(), 1).otherwise(0)).alias("with_weather"),
        F.sum(F.when(F.col("surname").isNotNull(), 1).otherwise(0)).alias("with_customer"),
        F.sum(F.when(F.col("region").isNotNull(), 1).otherwise(0)).alias("with_address"),
    ).first()
    print(f"{master_name}: {s['rows']:,} rows (renewed policies {n_pol:,} of {n_pol_all:,}, match: {s['rows'] == n_pol}) | "
          f"claims matched: {s['with_claims']:,} ({100.0 * s['with_claims'] / s['rows']:.1f}%) | "
          f"weather: {s['with_weather']:,} | customers: {s['with_customer']:,} | "
          f"addresses: {s['with_address']:,} | claim_indicator total: {s['claim_ind_total']:,} "
          f"(consistent with claims: {s['claim_ind_total'] == s['with_claims']}) | "
          f"earned_premium nulls: {s['earned_null']:,} | earned > written: {s['earned_over']:,} | "
          f"ultimate_claims total: {s['ult_total']:,.0f} (consistent: {s['ult_total'] == s['chk_net']})")

  

In [0]:
# Save the first 100 rows of each master policy file to its own Excel workbook
for name, master_df in master_dataframes.items():
    out_path = os.path.join(excel_dir, f"{name}_first_100_rows.xlsx")
    master_df.limit(100).toPandas().to_excel(out_path, engine="openpyxl", index=False)
    print(f"Saved {os.path.basename(out_path)} to: {excel_dir}")

In [0]:
# Examples to verify the earned_premium calculation (from the motor master file)
# earned_premium = written_premium (annual) x days in force / days in the calendar year
ex = (df_policies_motor_master
      .filter(F.col("period_start").isNotNull() & F.col("period_end").isNotNull() & F.col("written_premium").isNotNull())
      .withColumn("days_total", F.datediff("period_end", "period_start") + 1)
      .withColumn("year_days", F.datediff(
          F.make_date(F.year("period_start"), F.lit(12), F.lit(31)),
          F.make_date(F.year("period_start"), F.lit(1), F.lit(1))) + 1)
      .select("policy_id", "period_start", "period_end", "written_premium",
              "days_total", "year_days", "earned_premium"))

# Show a spread of real examples: January starts, mid-year starts, and very short periods
jan_full = ex.filter(F.month("period_start") == 1).orderBy(F.desc("days_total")).limit(2)
mid_year = ex.filter(F.month("period_start") == 7).limit(2)
short = ex.orderBy("days_total").limit(2)

print("Real motor policies (earned = written x days in force / days in year):")
display(jan_full)
display(mid_year)
display(short)

# Synthetic rows (NOT from the data) demonstrating the formula, incl. the 2024 leap year
syn = (spark.createDataFrame(
        [("TEST_FULL_2024", "2024-01-01", "2024-12-31", 500.0),   # full leap-year policy: 366/366
         ("TEST_90D_2021", "2021-07-01", "2021-09-28", 500.0)],  # 90-day policy: 90/365
        ["policy_id", "period_start", "period_end", "written_premium"])
       .withColumn("period_start", F.to_date("period_start"))
       .withColumn("period_end", F.to_date("period_end"))
       .withColumn("days_total", F.datediff("period_end", "period_start") + 1)
       .withColumn("year_days", F.datediff(
           F.make_date(F.year("period_start"), F.lit(12), F.lit(31)),
           F.make_date(F.year("period_start"), F.lit(1), F.lit(1))) + 1)
       .withColumn("earned_premium",
                   F.round(F.col("written_premium") * F.col("days_total") / F.col("year_days"), 2)))

print("Synthetic rows (not from the data):")
display(syn)
print("Hand check: TEST_FULL_2024 = 500.00 x 366/366 = 500.00 | TEST_90D_2021 = 500.00 x 90/365 = 123.29")

# Save the examples to Excel for review
ex_examples = (jan_full.unionByName(mid_year)
               .unionByName(short)
               .unionByName(syn.select("policy_id", "period_start", "period_end", "written_premium",
                                       "days_total", "year_days", "earned_premium"))).distinct()
ex_path = os.path.join(excel_dir, "earned_premium_examples.xlsx")
ex_examples.toPandas().to_excel(ex_path, engine="openpyxl", index=False)
print(f"\nSaved examples to: {ex_path}")

In [0]:
# Examples of the claims amount fields (from the motor master file)
# incurred_net_of_recoveries and ultimate_claims = incurred_to_date - recoveries
# (aggregated sums where a policy-year has multiple claims; null when no claims)
claim_cols = ["policy_id", "period_start", "n_claims", "incurred_to_date", "recoveries",
              "incurred_net_of_recoveries", "ultimate_claims"]

print("Single-claim rows:")
display(df_policies_motor_master.filter(F.col("n_claims") == 1).select(*claim_cols).limit(5))

print("Multi-claim rows (incurred/recoveries are sums across claims):")
display(df_policies_motor_master.filter(F.col("n_claims") > 1).select(*claim_cols).limit(5))

In [0]:
# Overview workbook for the three master policy files: one sheet per line
# (motor, contents, buildings), same layout as Data_overview_100_rows.xlsx
master_overview_path = os.path.join(excel_dir, "Master_Data_overview_100_rows.xlsx")

with pd.ExcelWriter(master_overview_path, engine="openpyxl") as writer:
    for name, master_df in master_dataframes.items():
        sheet = _sheet_name(name)   # e.g. policies_motor_master
        # Line at the top of the sheet stating the total number of rows
        summary = pd.DataFrame([[f"{name}: {master_df.count():,} total rows (showing first 100)"]])
        summary.to_excel(writer, sheet_name=sheet, index=False, header=False, startrow=0)
        # First 100 rows of the master dataframe, a few rows below the summary line
        master_df.limit(100).toPandas().to_excel(writer, sheet_name=sheet, index=False, startrow=3)

print(f"Master overview workbook saved to: {master_overview_path}")
print(f"Contains one sheet per master file: {', '.join(master_dataframes.keys())}")

In [0]:
import os
import pandas as pd

# Persist the three master policy files to workings/ as single-file parquet so any
# notebook (or a fresh session after a compute restart) can read them directly with
# spark.read.parquet(...) without re-running the data load + master build (~5 min).
# HUMAN DECISION (2026-10-20): masters are saved into the repo's workings/ folder on
# this branch. The raw data/ folder stays untouched (original Kaggle files).

workings_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/workings"
os.makedirs(workings_dir, exist_ok=True)

for name, master_df in master_dataframes.items():
    out_path = os.path.join(workings_dir, f"{name}.parquet")
    pdf_out = master_df.toPandas()
    pdf_out.attrs = {}   # Spark Connect attaches plan metrics to .attrs; pyarrow cannot serialize them
    pdf_out.to_parquet(out_path)   # single .parquet file (Git-friendly, no part files)

    # verify the round-trip with pandas (immediate). NOTE: a freshly written
    # /Workspace file can take a minute or more to become visible to spark.read.parquet
    # (workspace file-service propagation); Spark readability is checked separately.
    n_back = len(pd.read_parquet(out_path))
    print(f"{name}: {len(pdf_out):,} rows written | read back {n_back:,} rows | "
          f"match: {len(pdf_out) == n_back} | {out_path}")

#### Feature selection

In [0]:
# Feature selection for the motor Tweedie XGBoost — HUMAN DECISION (2026-10-20):
# expert-judgement first pass, to be revisited after EDA (see AI_ASSISTANCE.md).
# Target = ultimate_claims (nulls -> 0 for claim-free policy-years).
# 'Year' is derived from period_start. All claims-derived columns are excluded as
# features (leakage); IDs are kept only for GroupKFold grouping and prediction tracking.

MOTOR_FEATURES = [
    "Year",                                     # derived from period_start
    # exposure & premium (HUMAN DECISION 2026-10-20: written_premium is annualized —
    # use it directly; earned_premium dropped as a feature)
    "exposure", "written_premium",
    # driver & vehicle
    "driver_age", "years_licensed", "ncb_years", "vehicle_value_group", "vehicle_age",
    "make_model", "engine_size_cc", "body_type", "transmission", "seats", "fuel_type",
    "driver_assist", "cover_type", "annual_mileage", "overnight_parking", "vehicle_use",
    "named_young_driver", "telematics_opt_in", "telematics_score",
    "voluntary_excess", "compulsory_excess",
    # policy admin
    "channel", "payment_method", "missed_payments_12m", "new_business", "renewal_count",
    "days_quote_to_inception",
    # customer
    "occupation", "marital_status", "credit_band", "prior_claims_5y", "licence_points",
    "branch_id", "marketing_segment",
    # address (motor-relevant)
    "region", "urban_density",
    # weather (hail only for motor)
    "hail_days",
]

TARGET = "ultimate_claims"
GROUP_COL = "customer_id"              # GroupKFold grouping: all years of one customer stay together
TRACK_COLS = ["policy_id", "row_id"]  # for attaching predictions; never model inputs

# Every listed column must exist in the master (Year is derived here, not in the master)
_missing = [c for c in MOTOR_FEATURES + [TARGET, GROUP_COL, *TRACK_COLS]
            if c != "Year" and c not in df_policies_motor_master.columns]
assert not _missing, f"Listed columns missing from the master: {_missing}"

motor_model = (df_policies_motor_master
               .withColumn("Year", F.year("period_start"))
               # Exposure fix (HUMAN DECISION 2026-10-21): the raw exposure column is unreliable
               # (EDA: 0.47% of rows > 1, mixed day/year scales up to 365.0) — recompute from
               # the period dates as (period_end - period_start + 1) / 365
               .withColumn("exposure", (F.datediff("period_end", "period_start") + F.lit(1)) / F.lit(365))
               .withColumn(TARGET, F.coalesce(F.col(TARGET), F.lit(0.0)))   # no claims -> 0
               .select(*TRACK_COLS, GROUP_COL, *MOTOR_FEATURES, TARGET))

# One pass over the assembled dataset: size, target behaviour, feature missingness
s = motor_model.agg(
    F.count(F.lit(1)).alias("_rows"),
    F.sum(F.when(F.col(TARGET) > 0, 1).otherwise(0)).alias("_claim_rows"),
    F.round(F.sum(TARGET), 2).alias("_target_sum"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in MOTOR_FEATURES],
).first()

n_rows = s["_rows"]
print(f"Motor modeling dataset: {n_rows:,} rows | {len(MOTOR_FEATURES)} features | target = {TARGET} (nulls->0)")
print(f"Rows with target > 0: {s['_claim_rows']:,} ({100.0*s['_claim_rows']/n_rows:.1f}%) | total target sum: {s['_target_sum']:,.0f}")
print(f"Zero-target rows: {n_rows - s['_claim_rows']:,} ({100.0*(n_rows - s['_claim_rows'])/n_rows:.1f}%)")

feat_nulls = {c: s[c] for c in MOTOR_FEATURES if s[c] and s[c] > 0}
print("\nFeatures with missing values (left as-is — XGBoost handles them natively):")
for c, k in sorted(feat_nulls.items(), key=lambda x: -x[1]):
    print(f"  {c:<22} {100.0*k/n_rows:6.2f}%")

from collections import Counter
_dtypes = {c: t for c, t in motor_model.dtypes if c in MOTOR_FEATURES}
print(f"\nFeature dtypes: {dict(Counter(_dtypes.values()))}")

display(motor_model.limit(10))

#### Tweedie XGBoost modeling (motor)

4-fold GroupKFold (grouped by customer_id) Tweedie XGBoost on the motor master. The Tweedie variance power is fixed at 1.5 to match the hackathon scoring, and models are evaluated with mean Tweedie deviance at p=1.5 — the same loss form the competition uses.

In [0]:
%pip install xgboost openpyxl

In [0]:
import pandas as pd

# Convert the motor modeling dataset to pandas for XGBoost training:
# strings -> category (used natively via enable_categorical=True), booleans -> int.
# The same conversions, with category vocabularies learned from TRAINING only at
# scoring time, must be reused when scoring the test files so features stay aligned.

pdf = motor_model.toPandas()

CAT_FEATURES = [c for c in MOTOR_FEATURES if str(pdf[c].dtype) == "object"]
BOOL_FEATURES = [c for c in MOTOR_FEATURES if pdf[c].dtype == bool]
for c in CAT_FEATURES:
    pdf[c] = pdf[c].astype("category")
for c in BOOL_FEATURES:
    pdf[c] = pdf[c].astype("int8")

X = pdf[MOTOR_FEATURES]
y = pdf[TARGET].to_numpy(dtype=float)
w = pdf["exposure"].to_numpy(dtype=float)   # exposure weight: Tweedie pure premium model
groups = pdf[GROUP_COL]

print(f"Training matrix: {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"Categorical ({len(CAT_FEATURES)}): {', '.join(CAT_FEATURES)}")
print(f"Boolean -> int ({len(BOOL_FEATURES)}): {', '.join(BOOL_FEATURES)}")
print(f"Exposure weight: min {w.min():.4f} | max {w.max():.2f} | zero: {(w == 0).sum():,} | >1: {(w > 1).sum():,} (known data issue, EDA later)")
display(pdf.head(5))

In [0]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GroupKFold

# GroupKFold (4 folds): all policy-years of the same customer stay in one fold,
# so no customer is ever in both train and validation (avoids leakage across years).
# The hackathon scores with Tweedie power 1.5, so tweedie_variance_power is FIXED
# at 1.5 and each fold is evaluated with mean Tweedie deviance at p=1.5 — the same
# loss form the competition uses (XGBoost's built-in metric is a log-likelihood, not deviance).

POWER = 1.5          # Tweedie variance power — matches hackathon scoring (HUMAN NOTED)
N_FOLDS = 4

XGB_PARAMS = dict(
    objective="reg:tweedie",
    tweedie_variance_power=POWER,
    tree_method="hist",
    enable_categorical=True,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

def tweedie_deviance(y_true, y_pred, p=POWER):
    """Mean Tweedie unit deviance (1<p<2) — the loss form the hackathon scores on (p=1.5)."""
    y_true = np.maximum(np.asarray(y_true, dtype=float), 0.0)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 1e-9)
    unit = 2.0 * (
        np.power(y_true, 2.0 - p) / ((1.0 - p) * (2.0 - p))
        - y_true * np.power(y_pred, 1.0 - p) / (1.0 - p)
        + np.power(y_pred, 2.0 - p) / (2.0 - p)
    )
    return float(np.mean(unit))

gkf = GroupKFold(n_splits=N_FOLDS)
oof = pd.Series(np.nan, index=pdf.index, name="pred")
fold_rows = []
gain_total = {}   # feature -> summed gain across the 4 fold models

for k, (tr, va) in enumerate(gkf.split(X, y, groups=groups), start=1):
    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X.iloc[tr], y[tr], sample_weight=w[tr])
    pred = model.predict(X.iloc[va])
    oof.iloc[va] = pred

    score = model.get_booster().get_score(importance_type="gain")
    for f, g in score.items():
        gain_total[f] = gain_total.get(f, 0.0) + g

    dev = tweedie_deviance(y[va], pred)
    fold_rows.append({"fold": k, "train_rows": len(tr), "val_rows": len(va),
                      "tweedie_deviance_p1.5": dev,
                      "mean_pred": float(pred.mean()), "mean_actual": float(y[va].mean())})
    print(f"Fold {k}: {len(tr):,} train | {len(va):,} val | deviance(p=1.5) {dev:.4f} | "
          f"mean pred {pred.mean():.3f} vs actual {y[va].mean():.3f}")

fold_summary = pd.DataFrame(fold_rows)
print(f"\nMean Tweedie deviance (p=1.5) across folds: {fold_summary['tweedie_deviance_p1.5'].mean():.4f}")
display(fold_summary)

In [0]:
import os

# Out-of-fold (OOF) evaluation: every row is scored by a fold model that never saw it.
# Decile check: rows sorted into deciles by predicted pure premium — actual claims and
# claim rate should rise across deciles. Feature importance (gain) feeds the EDA/refinement
# loop. All results exported to Excel for review.

excel_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files"

pdf["oof_pred"] = oof
pdf["claim_flag"] = (pdf[TARGET] > 0).astype(int)
pdf["decile"] = pd.qcut(pdf["oof_pred"], 10, labels=False, duplicates="drop") + 1

decile_check = (pdf.groupby("decile")
    .agg(rows=("oof_pred", "size"),
         mean_pred=("oof_pred", "mean"),
         mean_actual=(TARGET, "mean"),
         claim_rate=("claim_flag", "mean"),
         total_actual=(TARGET, "sum"))
    .reset_index())

mono_share = (decile_check["mean_actual"].diff().dropna() >= 0).mean()
lift = decile_check.loc[decile_check["decile"].idxmax(), "mean_actual"] / pdf[TARGET].mean()
print(f"Decile monotonicity (share of steps non-decreasing in actual claims): {mono_share:.0%}")
print(f"Top-decile lift (mean actual claims, decile 10 vs overall): {lift:.2f}x")
display(decile_check)

feat_imp = (pd.DataFrame([{"feature": f, "gain": g} for f, g in gain_total.items()])
            .sort_values("gain", ascending=False).reset_index(drop=True))
feat_imp["gain_share_%"] = 100.0 * feat_imp["gain"] / feat_imp["gain"].sum()
display(feat_imp.head(15))

results_path = os.path.join(excel_dir, "motor_tweedie_xgb_results.xlsx")
with pd.ExcelWriter(results_path, engine="openpyxl") as writer:
    fold_summary.to_excel(writer, sheet_name="Fold_summary", index=False)
    decile_check.to_excel(writer, sheet_name="Decile_check", index=False)
    feat_imp.to_excel(writer, sheet_name="Feature_importance", index=False)
print(f"Results saved to: {results_path}")

#### Test-file scoring & first submission (motor)

Refit the motor Tweedie XGBoost on 100% of the training rows (the CV run was evaluation only), build the 2025 test master with the same joins as training, predict ultimate_claims, and write the submission CSV on the sample_submission skeleton. Buildings/contents rows (and any zero predictions) use a dummy value of 10 — the scorer does not accept zeros.

In [0]:
# Build the motor TEST master for the 2025 policies, mirroring the training master
# joins: weather (address_id + policy year), customers (customer_id), addresses
# (address_id). The test files have no claims data and no renewal_decision, so there
# is no claims join and NO renewed-only filter — every test row is scored (population
# mismatch vs the renewed-only training filter is a known flagged decision).

wx = (dataframes["weather"]
      .withColumnRenamed("address_id", "_wx_address_id")
      .withColumnRenamed("year", "_wx_year"))

test_motor_master = (
    dataframes["test_motor"]
    .join(wx,
          (F.col("address_id") == F.col("_wx_address_id")) &
          (F.year(F.col("period_start")) == F.col("_wx_year")),
          "left")
    .drop("_wx_address_id", "_wx_year")
    .join(dataframes["customers"], "customer_id", "left")
    .join(dataframes["addresses"], "address_id", "left")
)

test_motor_model = (test_motor_master
    .withColumn("Year", F.year("period_start"))
    # Same exposure fix as training (HUMAN DECISION 2026-10-21): recompute from the
    # period dates as (period_end - period_start + 1) / 365, not the raw column
    .withColumn("exposure", (F.datediff("period_end", "period_start") + F.lit(1)) / F.lit(365))
    .select("row_id", "policy_id", "customer_id", *MOTOR_FEATURES))

# Every training feature must be present in the test master
_t_missing = [c for c in MOTOR_FEATURES if c != "Year" and c not in test_motor_model.columns]
assert not _t_missing, f"Features missing from the test master: {_t_missing}"

test_pdf = test_motor_model.toPandas()
print(f"Test motor master: {len(test_pdf):,} rows | {len(MOTOR_FEATURES)} features (same as training)")
print(f"Customer-feature nulls (join gap): {test_pdf['occupation'].isna().mean():.1%}")

# Align categoricals with the TRAINING vocabularies; unseen test levels -> NaN.
# Booleans -> int, exactly as in the training prep.
X_test = test_pdf[MOTOR_FEATURES].copy()
_new_levels = {}
for c in CAT_FEATURES:
    cats = pdf[c].cat.categories
    _new_levels[c] = int((~test_pdf[c].dropna().isin(cats)).sum())
    X_test[c] = pd.Categorical(test_pdf[c], categories=cats)
for c in BOOL_FEATURES:
    X_test[c] = X_test[c].astype("int8")

_new = {c: k for c, k in _new_levels.items() if k > 0}
print(f"Unseen categorical levels (mapped to NaN): {_new if _new else 'none'}")
display(X_test.head(5))

In [0]:
import os
import numpy as np

# FINAL model: refit the winning configuration on 100% of the training rows — the CV
# run was evaluation only (mean deviance 107.8 at p=1.5 with the fixed exposure
# column); the final model uses all training data before scoring the 2025 test rows.
final_model = xgb.XGBRegressor(**XGB_PARAMS)
final_model.fit(X, y, sample_weight=w)
test_pred = final_model.predict(X_test)

# Submission on the sample_submission skeleton (row_id + prediction), following the
# team's submissions/ convention with 2-dp rounding. Buildings/contents rows have no
# model yet and the scorer rejects zeros, so they — and any rounded-zero motor
# prediction — get the dummy value 10 (HUMAN INSTRUCTION 2026-10-20).
DUMMY = 10
assert test_pdf["row_id"].is_unique, "test_motor row_id is not unique"

sub = dataframes["sample_submission"].select("row_id").toPandas()
motor_map = pd.DataFrame({"row_id": test_pdf["row_id"], "prediction": np.round(test_pred, 2)})
sub = sub.merge(motor_map, on="row_id", how="left")
sub["prediction"] = sub["prediction"].fillna(DUMMY)
sub.loc[sub["prediction"] == 0, "prediction"] = DUMMY

# Sanity checks: full skeleton, no nulls or zeros, motor fully scored
assert len(sub) == 626_639, f"unexpected row count: {len(sub):,}"
assert sub["prediction"].isna().sum() == 0 and (sub["prediction"] <= 0).sum() == 0

pref = sub["row_id"].str[0]
motor_mask = pref == "M"
print(f"Submission rows: {len(sub):,} | by line: {pref.value_counts().to_dict()}")
print(f"Motor rows scored by the model: {motor_mask.sum():,} | "
      f"mean pred {sub.loc[motor_mask, 'prediction'].mean():.2f} | "
      f"min {sub.loc[motor_mask, 'prediction'].min():.2f} | "
      f"max {sub.loc[motor_mask, 'prediction'].max():.2f}")
print(f"Dummy rows (B/C lines, no model yet): {(~motor_mask).sum():,} | "
      f"motor zero-preds replaced by dummy: {int(sub.loc[motor_mask, 'prediction'].eq(DUMMY).sum())}")

sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"
os.makedirs(sub_dir, exist_ok=True)
sub_path = os.path.join(sub_dir, "motor_tweedie_xgb_submission_v002.csv")
sub.to_csv(sub_path, index=False)
print(f"\nSubmission saved to: {sub_path}")
display(sub.head(10))

In [0]:
import os
import pandas as pd

# Combined M + B + C submission v006: motor v002 predictions (fixed exposure), with
# contents v001 and buildings v002 predictions carried over unchanged from v005.
sub_dir = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/submissions"

motor_v2 = pd.read_csv(os.path.join(sub_dir, "motor_tweedie_xgb_submission_v002.csv"))
combined_v5 = pd.read_csv(os.path.join(sub_dir, "motor_buildings_contents_submission_v005.csv"))

# Same skeleton, same order — just swap the M-row predictions
assert len(motor_v2) == len(combined_v5) == 626_639, "unexpected row counts"
assert (motor_v2["row_id"] == combined_v5["row_id"]).all(), "row_id mismatch between files"

combined = combined_v5.copy()
is_motor = combined["row_id"].str[0] == "M"
combined.loc[is_motor, "prediction"] = motor_v2.loc[is_motor, "prediction"].values

# Sanity checks: full skeleton, no nulls or non-positive predictions
assert combined["prediction"].isna().sum() == 0
assert (combined["prediction"] <= 0).sum() == 0

pref = combined["row_id"].str[0]
print(f"Combined submission rows: {len(combined):,} | by line: {pref.value_counts().to_dict()}")
for line in ["M", "C", "B"]:
    p = combined.loc[pref == line, "prediction"]
    print(f"  {line}: {len(p):,} rows | mean {p.mean():.2f} | min {p.min():.2f} | max {p.max():.2f}")

out_path = os.path.join(sub_dir, "motor_buildings_contents_submission_v006.csv")
combined.to_csv(out_path, index=False)
print(f"\nSaved combined submission to: {out_path}")
display(combined.head(10))